# L68 · CTC（Connectionist Temporal Classification，CTC） 对齐（alignment）原理 — blank 符号、单调路径与标签折叠（label collapse）

**目标**：理解 CTC 的核心思想——blank 符号 + 去重规则 + 路径求和，让模型在无帧级对齐标注的情况下完成序列训练。

🔗 **Aurora 连接**：`aurora.speech` 将调用 `torch.nn.CTCLoss`；读懂 CTC 是理解 Whisper 训练目标的前提。 本节手写贪婪解码，建立对 CTC 路径压缩的直觉。

语音识别的根本难题：模型每帧输出一个 token，但目标文字序列比帧数短得多，且不知道每个字符对应哪几帧。
CTC 通过引入 **blank（∅）** 符号，把所有「压缩后与目标相同」的帧级序列（路径）的概率加起来作为训练信号——
这样模型只需要知道「最终输出什么」，不需要知道「每帧输出什么」。

**blank 有两个不可缺少的作用**：

1. **表示"还没决定"**：模型每帧必须输出一个 token，但语音远比字符密集（约 50–100 帧/字符）。blank 允许模型说"这一帧我暂时不承诺"，等到声学特征积累充分后再给出字符。

2. **分隔重复字母（唯一方案）**：这是 blank 最关键的作用。
   - 目标词 `hello` 含两个相邻 `l`。路径 `h e l l o` 去重后变成 `h e l o`，**错误**。
   - 正确路径：`h e l ∅ l o`——blank 打断了 `ll`，折叠后保留两个 `l`。
   - 没有 blank，CTC 就无法区分 `hello` 和 `helo`；blank 是分隔重复字母的**唯一**手段。

In [ ]:
import numpy as np
import torch

## 1. CTC 路径与路径概率

设输入序列长度为 `T`，词表（vocabulary）大小为 `V`（含 blank=0）。
模型在每一帧 `t` 输出一个概率分布 `p(token | t)`，形状 `(T, V)`。

**CTC 路径** `π` 是长度为 `T` 的 token 序列（每帧一个符号，含 blank）。
路径概率为各帧概率之积：

```
P(π) = ∏_{t=1}^{T}  p(π_t | t)
```

**CTC 损失** 最大化所有「压缩后等于目标 `y`」的路径的概率之和：

```
P(y | x) = Σ_{π : collapse(π)=y}  P(π)
```

路径数量是指数级的，实际用**前向-后向算法**动态规划（dynamic programming，DP）精确计算。

In [ ]:
# 演示：3帧，词表={blank=0, a=1, b=2}
# 构造一个简单的对数概率矩阵（手动指定）
T, V = 3, 3
log_probs = np.array([
    [np.log(0.1), np.log(0.8), np.log(0.1)],  # 帧0：a 最大
    [np.log(0.7), np.log(0.2), np.log(0.1)],  # 帧1：blank 最大
    [np.log(0.1), np.log(0.1), np.log(0.8)],  # 帧2：b 最大
])
# 列举路径 (a, blank, b) 和 (a, a, b) 两条路径，两者压缩后都是 [a, b]
path1 = [1, 0, 2]  # a ∅ b  -> collapse -> a b
path2 = [1, 1, 2]  # a a b  -> collapse -> a b
p_path1 = np.exp(log_probs[0,1] + log_probs[1,0] + log_probs[2,2])
p_path2 = np.exp(log_probs[0,1] + log_probs[1,1] + log_probs[2,2])
print(f'P(a ∅ b) = {p_path1:.4f}')
print(f'P(a a b) = {p_path2:.4f}')
print(f'P(y=[a,b] | x) ≈ {p_path1 + p_path2:.4f}  (只枚举了两条路径)')


## 2. 压缩规则（collapse）

给定一条路径 `π`（含 blank），压缩为目标序列的规则分两步：

```
步骤1：去掉相邻重复  [a, a, ∅, b, b, ∅, c] → [a, ∅, b, ∅, c]
步骤2：去掉 blank    [a, ∅, b, ∅, c]       → [a, b, c]
```

注意 blank 的作用之一是**分隔同字符**：
路径 `[a, a, b]` 压缩为 `[a, b]`；
路径 `[a, ∅, a, b]` 压缩为 `[a, a, b]`（两个 a 之间有 blank 则不合并）。

In [ ]:
def collapse(path, blank=0):
    """CTC 路径压缩：先去相邻重复，再去 blank。"""
    deduped = [p for i, p in enumerate(path) if i == 0 or p != path[i-1]]
    return [p for p in deduped if p != blank]

# 示例验证
examples = [
    ([1, 1, 0, 2, 2, 0, 3], 'a a ∅ b b ∅ c'),
    ([1, 0, 1, 2],           'a ∅ a b'),
    ([1, 1, 2],              'a a b'),
    ([0, 0, 1, 0],           '∅ ∅ a ∅'),
]
for path, desc in examples:
    result = collapse(path)
    print(f'  [{desc}]  →  {result}')


## 3. 贪婪解码（Greedy Decode）

精确 CTC 解码需要 beam search（枚举所有路径太慢）。
**贪婪解码**是最快的近似：每帧直接取 argmax，得到一条路径，再执行 collapse。

```
π_greedy = [argmax p(token | t)  for t in 1..T]
y_greedy = collapse(π_greedy)
```

贪婪解码不是最优的——它只考虑了最高概率的单条路径，而正确答案可能由多条次优路径联合撑起来。
但在 Whisper、DeepSpeech 等实践中，它作为快速 baseline 效果已经不错。

In [ ]:
# 演示贪婪解码
np.random.seed(42)
T_demo, V_demo = 7, 5  # 7帧，词表大小5（0=blank, 1=a, 2=b, 3=c, 4=d）
logits_demo = np.random.randn(T_demo, V_demo)
# 手动拉高某些帧让结果更直观
logits_demo[0, 1] = 5   # 帧0: a
logits_demo[1, 0] = 5   # 帧1: blank
logits_demo[2, 0] = 5   # 帧2: blank
logits_demo[3, 2] = 5   # 帧3: b
logits_demo[4, 2] = 5   # 帧4: b
logits_demo[5, 0] = 5   # 帧5: blank
logits_demo[6, 3] = 5   # 帧6: c

preds = np.argmax(logits_demo, axis=1)
vocab = {0:'∅', 1:'a', 2:'b', 3:'c', 4:'d'}
print('帧级 argmax  :', [vocab[p] for p in preds])
print('去相邻重复  :', [vocab[p] for i,p in enumerate(preds) if i==0 or p!=preds[i-1]])
decoded = collapse(preds.tolist())
print('最终解码结果:', [vocab[p] for p in decoded], '→ token ids:', decoded)


## 4. ✏️ 实现 `ctc_greedy_decode(logits, blank=0)`

**推理路线**：
1. `logits` 形状 `(T, vocab_size)`，argmax 不依赖 softmax，直接比大小
2. `preds = np.argmax(logits, axis=1)` → shape `(T,)`，每帧一个 token id
3. 去相邻重复：`deduped = [p for i,p in enumerate(preds) if i==0 or p!=preds[i-1]]`
4. 去 blank：`[p for p in deduped if p != blank]`，返回 `list[int]`

**参考输入输出**：
```
# 7帧，logits 让每帧 argmax 依次为 [a=1, ∅=0, ∅=0, b=2, b=2, ∅=0, c=3]
# 去重 → [1, 0, 2, 0, 3]
# 去blank → [1, 2, 3]  即 [a, b, c]
ctc_greedy_decode(logits)  # → [1, 2, 3]
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def ctc_greedy_decode(logits: np.ndarray, blank: int = 0) -> list:
    """CTC 贪婪解码：argmax 每帧，去相邻重复，去 blank。

    Args:
        logits: shape (T, vocab_size)，未经 softmax 的原始分数
        blank:  blank 符号的 token id，默认 0
    Returns:
        解码后的 token id 列表
    """
    # ✏️ TODO: 第1步——每帧取 argmax
    preds = None

    # ✏️ TODO: 第2步——去相邻重复（保留第一个，后续只保留与前一个不同的）
    deduped = None

    # ✏️ TODO: 第3步——去掉 blank，返回 list[int]
    return None


In [ ]:
# --- 检查 ctc_greedy_decode ---

# 构造 7 帧 logits，让 argmax 依次为 [1,0,0,2,2,0,3]（a ∅ ∅ b b ∅ c）
V_check = 5
logits_check = np.full((7, V_check), -10.0)
forced_ids = [1, 0, 0, 2, 2, 0, 3]
for t, tok in enumerate(forced_ids):
    logits_check[t, tok] = 10.0

result = ctc_greedy_decode(logits_check, blank=0)
if result is None:
    print('⬜ 请先实现 ctc_greedy_decode 的三个 TODO 步骤')
else:
    assert result == [1, 2, 3], f'期望 [1,2,3]，实际得到 {result}'
print('✅ ctc_greedy_decode([a ∅ ∅ b b ∅ c]) =', result, '= [a, b, c]')

# 边界：全 blank 输入 → 空列表
logits_blank = np.zeros((5, V_check))
logits_blank[:, 0] = 10.0
assert ctc_greedy_decode(logits_blank) == [], '全 blank 应返回空列表'
print('✅ 全 blank 输入 → []')

# 边界：单帧单字符
logits_single = np.array([[-1.0, 5.0, -1.0]])
assert ctc_greedy_decode(logits_single) == [1]
print('✅ 单帧 argmax=1 → [1]')


## 5. 参数实验：随机 logits 下的序列压缩率

**实验设置**：
- `T = 50`（输入帧数），`vocab_size = 30`（含 blank=0）
- 跑 1000 次，每次随机采样 `logits ~ N(0,1)`
- 统计 `len(decoded) / T` 的均值和分布

**预期现象**：
- 每帧随机时，blank 被选中概率约 `1/30 ≈ 3.3%`，相邻重复概率约 `1/30`
- 去重+去blank 后，平均输出长度约为输入的 `~65-70%`
- 实际语音模型训练后 blank 被大量预测，压缩率会远高于此随机基线

In [ ]:
np.random.seed(0)
T_exp, V_exp, N = 50, 30, 1000

ratios = []
for _ in range(N):
    logits_exp = np.random.randn(T_exp, V_exp)
    decoded_exp = ctc_greedy_decode(logits_exp, blank=0)
    ratios.append(len(decoded_exp) / T_exp)

ratios = np.array(ratios)
print(f'输入长度 T = {T_exp}，词表大小 V = {V_exp}（含blank）')
print(f'1000次随机实验：')
print(f'  平均输出长度     : {ratios.mean()*T_exp:.1f} 帧')
print(f'  平均压缩率 len/T : {ratios.mean():.3f}')
print(f'  最小/最大压缩率  : {ratios.min():.3f} / {ratios.max():.3f}')
print()
print('直觉：随机时每帧选到 blank 概率 1/V ≈', round(1/V_exp, 3),
      '；相邻字符相同概率同量级，因此压缩率 ≈ 1 - 1/V - 去重贡献')


## 本课收束

本节实现了 `ctc_greedy_decode`，它输出一个 `list[int]`（压缩后的 token id 序列），
对应 CTC 路径的近似最优解。
该函数直接服务于 `aurora.speech` 推理管线——在 `torch.nn.CTCLoss` 训练完成后，
贪婪解码是最快的线上解码方式。
下一节（L69）将实现 CTC 前向算法的完整动态规划，追踪 blank-skip 和标签折叠两条合法路径，计算序列概率。
以及为什么 Whisper 用交叉熵而非 CTC 作为训练目标。